# Long Context Extrapolation

> LLaMA 2 was trained on sequences of only 4096 tokens. Yet today's models handle 128K or even 1M tokens — same architecture, lengths never seen during training, and inference still works.
>
> The answer lies in position encoding extrapolation techniques. This section starts from Attention's length bottleneck, works through why RoPE extrapolates, what PI does, how NTK and YaRN improve upon it, and finally how to test long-context capability.

Long context extrapolation solves one concrete problem: the model has only seen position encodings for positions 0 through 4095 during training, but at inference time it must process position 10000 — a position encoding the model has never encountered.

The solution is not retraining. Instead, we make the position encoding patterns learned during training continue to work on longer sequences. The specific method depends on the position encoding design: RoPE encodes relative positions through rotation matrices and has some natural extrapolation ability; PI compresses position indices; NTK adjusts the frequency base; YaRN applies differentiated treatment to different frequency dimensions. These four methods form a progressive optimization path, which is the main thread of this section.

## 1. What is Extrapolation

**Extrapolation** = using patterns from a known range to predict what happens outside that range.

A real-life example:
- You have measured water temperature: 0°C → ice, 50°C → liquid, 100°C → boiling
- Now someone asks: what happens to water at 200°C? You haven't measured it, but based on the pattern you can infer → still gas
- That is extrapolation

LLMs face the same problem:
- During training: the model learned attention patterns for positions 0 to 4095
- During inference: the user provides a 10000-token article
- The question: can the model correctly handle tokens at positions 4096–9999, which it never saw during training?

**The answer depends on which position encoding you use.**

## 2. Position Encoding Review

(If you are already familiar with Embedding + Position from Part 3, you can skip this section. But RoPE later builds on this foundation, so if unsure it's worth a quick review.)

Attention itself is **order-agnostic**. Feed "cat sat mat" and "mat sat cat" to attention, and it computes the same attention scores — because attention only looks at "how related" tokens are, not "who comes before whom."

But order clearly matters. "I love you" and "you love I" mean completely different things.

**Position encoding attaches an "I am the Nth token" label to each token**, so that attention can use this information when computing relevance.

There are three ways to attach these labels:

In [ ]:
# Intuition: how order affects attention
print("Sentence A: I love you")
print("Sentence B: you love I")
print()
print("Without position encoding:")
print("  'I' and 'you' have the same attention score regardless of order")
print("  The model cannot distinguish 'I love you' from 'you love I'")
print()
print("With position encoding:")
print("  'I' at position 0 and position 2 gets different vectors → attention can distinguish them")
print()
print("Here's the problem: training only saw up to 4096 positions,")
print("but inference receives 10000 positions → what does the label for position 4097 look like?")

## 3. Extrapolation Capability of Three Position Encodings

| Method | How it works | Representative model | Can extrapolate? | Why? |
|------|--------|---------|----------|--------|
| **Learned positions** | Randomly initialize a vector for each position during training, adjust during training | GPT-2 | No, not at all | Only learned vectors for positions 0–1023; the vector for position 1024 simply doesn't exist |
| **Sinusoidal encoding** | Use sin/cos functions to hand-compute each position's value, no learning needed | Original Transformer | Theoretically yes, practically poor | The functions are continuous, but the model hasn't learned to exploit that continuity |
| **RoPE (Rotary Position Encoding)** | Encode positions via "rotation"; position difference = rotation angle difference | LLaMA, Qwen, Mistral | Yes | Relative positions are naturally extrapolatable, and frequency-domain properties can be exploited |

RoPE is now standard in nearly all open-source LLMs. Let's understand it.

## 4. Intuition for RoPE

Imagine a clock. Not an ordinary 12-hour clock, but a "position clock":

- Token at position 0 → hand points to 12 o'clock (0°)
- Token at position 1 → hand rotates clockwise a bit (say 30°)
- Token at position 2 → hand rotates a bit more (60°)
- ...

**Key insight: RoPE doesn't add a value to each position — it rotates the token's vector.**

```
Sinusoidal: token vector + position vector = final vector (addition)
RoPE:       rotate token vector by an angle     = final vector (rotation)
```

Why is rotation better? **Because the difference in rotation angles equals the relative position.**

After two vectors are rotated and then dotted, the result depends only on the **difference** in their rotation angles:
- Position 0 and position 1: angle difference 30° → dot product = cos(30°)
- Position 5 and position 6: angle difference is also 30° → dot product = also cos(30°)
- Position 0 and position 3: angle difference 90° → dot product = cos(90°)

**Two adjacent tokens, whether at the beginning or end of the sentence, have the same attention between them!** This property is crucial.

## 5. RoPE Computation Steps

Without skipping any steps.

#### Step 1: Split the vector into pairs of "2D vectors"

A token's vector has d_k dimensions, say 64. We group them in pairs: (dim 0, dim 1), (dim 2, dim 3), ..., (dim 62, dim 63). That's 32 pairs total.

Each pair is a 2D coordinate `(x, y)`. **Rotation happens in this 2D plane.**

#### Step 2: Each pair of dimensions has a different rotation speed

Pair 0 rotates **fast** (a big step per position) → responsible for distinguishing adjacent positions
Pair 31 rotates **slow** (a tiny step per position) → responsible for long-range information

This is like a clock:
- **Second hand** (high frequency): rotates fast, distinguishes "just now" from "now"
- **Minute hand** (medium frequency): rotates slower, distinguishes "a few minutes ago"
- **Hour hand** (low frequency): rotates slowest, distinguishes "a few hours ago"

Mathematically, the rotation speed (frequency) of pair i is:
```
freq_i = 1 / base^(2i/d)
```
where `base` defaults to 10000. Larger i means larger denominator means smaller frequency → slower rotation.

In [ ]:
# Direct look: how much do different dimension pairs' rotation speeds differ?
import torch
import math

d_k = 64          # 64 dimensions total, paired → 32 pairs
base = 10000      # RoPE default base

pair_indices = torch.arange(0, d_k, 2).float()  # [0, 2, 4, ..., 62]
freqs = 1.0 / (base ** (pair_indices / d_k))

print(f"Total: {len(freqs)} dimension pairs")
print(f"Pair 0 (fastest) frequency: {freqs[0]:.4f}  → rotates {math.degrees(freqs[0]):.1f}° per position")
print(f"Pair 16 (medium) frequency: {freqs[16]:.6f}  → rotates {math.degrees(freqs[16]):.4f}° per position")
print(f"Pair 31 (slowest) frequency: {freqs[31]:.8f}  → rotates {math.degrees(freqs[31]):.6f}° per position")

slowest_period = 2 * math.pi / freqs[31]
print(f"\nThe slowest pair needs {slowest_period:.0f} positions to complete one full rotation")
# → Training window is 4096; the slowest hand hasn't even completed one rotation → this is the extrapolation bottleneck

In [ ]:
# Visualize: different dimensions' hands moving with position (cos values)
import torch
import matplotlib.pyplot as plt

seq_len = 200
positions = torch.arange(seq_len).float()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax_idx, (pair_idx, label) in enumerate([
    (0, "Fast (second hand)"),
    (16, "Medium (minute hand)"),
    (31, "Slow (hour hand)")
]):
    theta = positions * freqs[pair_idx]
    ax = axes[ax_idx]
    ax.plot(positions.numpy(), theta.cos().numpy(), linewidth=1)
    ax.set_xlabel('Position'); ax.set_ylabel('cos(angle)')
    ax.set_title(f'Pair {pair_idx} — {label}\n{math.degrees(freqs[pair_idx]):.2f} deg per step')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Left: several full rotations (dense waveform) → high frequency → distinguishes neighbors
# Right: less than half a rotation (gentle curve) → low frequency → carries long-range information

## 6. Why Direct Extrapolation Fails

The training window is 4096. Now at inference we get 8192 tokens. **Can we just tell RoPE to keep counting: position 4097, 4098, ...?**

No. Look at the low-frequency dimension (the slowest hand):

- During training: it only rotated within the range 0–4095, so it saw cos(θ) where θ is in [0, some range]
- At inference reaching position 8192: θ = 8192 × freq, this θ value exceeds the range seen during training
- The model has no experience with the out-of-range θ values → attention goes wrong → output is garbled

**Analogy: you've only learned sin(0°) to sin(45°), and now someone asks you to compute sin(180°) — you're stuck.**

In [ ]:
# The problem with direct extrapolation: low-frequency dimensions exceed trained angle range
import torch
import matplotlib.pyplot as plt
import math

train_len, extrap_len = 4096, 8192
slow_pair = 31

positions_train = torch.arange(train_len).float()
positions_extrap = torch.arange(extrap_len).float()
theta_train = positions_train * freqs[slow_pair]
theta_extrap = positions_extrap * freqs[slow_pair]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(positions_extrap.numpy(), theta_extrap.cos().numpy(),
        linewidth=1, color='orange', label='cos value at inference')
ax.axvline(x=train_len, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.fill_between(range(train_len, extrap_len), -1.2, 1.2,
                alpha=0.1, color='red', label='unseen angle range')
ax.set_xlabel('Position'); ax.set_ylabel('cos(angle)')
ax.set_title(f'Direct extrapolation to 8192 (low-frequency dim #{slow_pair})\nRight of red line = unseen angle range')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

# Max angle during training vs. after extrapolation
train_deg = math.degrees(theta_train[-1].item())
extrap_deg = math.degrees(theta_extrap[-1].item())
print(f"Max angle during training: {train_deg:.1f}°  →  Max angle after extrapolation: {extrap_deg:.1f}°  →  Exceeded by {extrap_deg - train_deg:.1f}°")

## 7. Core Idea: Angle Mapping

Since the model only recognizes angles corresponding to positions 0–4095, we can make positions 4096–8191 produce angles that fall within the training range.

```
Original RoPE (direct extrapolation → fails):
  Position 0    → angle 0°         ✅ model recognizes
  Position 4096 → angle 1000°      ❌ doesn't recognize! never learned
  Position 8192 → angle 2000°      ❌ even less recognizable!

Extrapolated RoPE (angle compression → succeeds):
  Position 0    → angle 0°         ✅ recognizes
  Position 4096 → angle 500°       ✅ recognizes! (compressed into training range)
  Position 8192 → angle 1000°      ✅ recognizes!
```

**The core operation in one sentence: through some method, make the rotation angles produced by very long positions not exceed the angle range seen during training.**

The remaining question is: what is the most reasonable way to compress? Let's look at three methods.

## 8. Method 1: Position Interpolation

**Paper**: Meta, 2023 — Extending Context Window via Position Interpolation

Idea: directly **proportionally compress** position indices.

```
Goal: extend 4096 window to 8192
Scaling factor α = 4096 / 8192 = 0.5

New position = real position × 0.5

Real position 0    → position given to model = 0 × 0.5 = 0
Real position 2048 → position given to model = 2048 × 0.5 = 1024
Real position 8192 → position given to model = 8192 × 0.5 = 4096 ← exactly at the training boundary!
```

**Analogy**: your street has house numbers 1 to 100, but you only recognize numbers 1–50. Now numbers 51–100 arrive, and you divide all numbers by 2 — number 51 becomes 25.5, number 100 becomes 50, all within your recognized range.

**Cost**: all house numbers are compressed. Originally you could clearly distinguish number 1 from number 2; now 1 and 2 become 0.5 and 1 — the difference shrinks. Local resolution drops.

In [ ]:
# PI implementation: position × scaling factor, then compute RoPE normally
import torch
import matplotlib.pyplot as plt

train_len, target_len = 4096, 8192
alpha = train_len / target_len  # 0.5

pair_indices = torch.arange(0, 64, 2).float()
freqs_orig = 1.0 / (10000 ** (pair_indices / 64))
freqs_pi = freqs_orig * alpha

# Original RoPE vs PI compressed waveform
positions_orig = torch.arange(target_len).float()
angles_orig = positions_orig * freqs_orig[31]
positions_pi = torch.arange(target_len).float() * alpha
angles_pi = positions_pi * freqs_orig[31]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(angles_orig.cos().numpy(), linewidth=1, label='Original RoPE')
axes[0].plot(angles_pi.cos().numpy(), linewidth=1, label=f'PI (×{alpha:.2f})')
axes[0].axvline(x=train_len, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Position'); axes[0].set_ylabel('cos(angle)')
axes[0].set_title(f'Low-frequency dim #{31} wave\nPI stretches the wave (half frequency)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: all dimension frequencies are compressed proportionally
axes[1].plot(freqs_orig.numpy(), 'o-', markersize=3, label='Original frequency')
axes[1].plot(freqs_pi.numpy(), 's-', markersize=3, label='After PI scaling')
axes[1].set_xlabel('Dimension pair index (0=fast, 31=slow)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('PI scales all dimensions equally\nLocal resolution drops, light tuning needed')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Method 2: NTK-aware

**Paper**: NTK-Aware Scaled RoPE, bloc97, 2023

PI's flaw: it compresses **all** dimensions equally. But from Section 5, we know different dimensions rotate at different speeds:
- Fast dimensions (second hand): already completed many rotations within 4096 positions, so they've seen all kinds of angles → **no compression needed**
- Slow dimensions (hour hand): haven't even completed one rotation within 4096 positions, many unseen angles → **compression needed**

So NTK-aware's idea is: **only compress the slow ones, leave the fast ones alone.**

How? **Increase the base from 10000.** This is NTK-aware's most elegant insight:

```
Frequency formula: freq_i = 1 / base^(2i/d)

base = 10000 → fast frequency → slow dimension can't complete one rotation
base = 100000 → frequency slows → slow dimension rotates even slower → smaller angles within same positions → doesn't exceed training range!

Moreover:
  Low i (fast dimensions): freq ≈ 1 → changing base barely affects them ← fast ones don't need adjustment
  High i (slow dimensions): freq ≈ 1/base → changing base has large effect ← slow ones get adjusted a lot
```

This precisely achieves "don't adjust fast hands, adjust slow hands a lot"! **Changing one parameter automatically accomplishes differentiated compression.**

In [ ]:
# Demonstrate NTK: effect of changing base on different dimensions
import torch
import matplotlib.pyplot as plt

base_old, scale = 10000, 2
# NTK formula: new base = old base × scale^(d/(d-2))
base_new = base_old * (scale ** (64 / 62))

pair_indices = torch.arange(0, 64, 2).float()
freqs_old = 1.0 / (base_old ** (pair_indices / 64))
freqs_new = 1.0 / (base_new ** (pair_indices / 64))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(freqs_old.numpy(), 'o-', markersize=3, label=f'base={base_old}')
axes[0].plot(freqs_new.numpy(), '^-', markersize=3, label=f'base={base_new:.0f}')
axes[0].set_xlabel('Dimension pair index (0=fast, 31=slow)'); axes[0].set_ylabel('Frequency')
axes[0].set_title('NTK-aware: increase base\nFast dims stay similar, slow dims slow down')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Compression ratio for each dimension
ratio = freqs_new / freqs_old
axes[1].bar(range(len(ratio)), ratio.numpy())
axes[1].set_xlabel('Dimension pair index'); axes[1].set_ylabel('New frequency / old frequency')
axes[1].set_title('Compression ratio by dimension\nFast dims ~100%, slow dims ~50%')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# NTK only needs to change base from 10000 to ~86000, no model structure change, and most cases need no fine-tuning

## 10. Method 3: YaRN

**Paper**: YaRN, 2023

NTK is already quite good, but YaRN identified one issue: after changing the base, the middle dimensions (neither fast nor slow) have attention that becomes "less decisive."

What does that mean? Recall the softmax step in attention:

```
softmax([2, 1, 0.5]) → [0.59, 0.22, 0.13, 0.06]  ← relatively "sharp", attention is concentrated
softmax([1, 0.5, 0.25]) → [0.42, 0.26, 0.19, 0.14] ← relatively "flat", attention is dispersed
```

Using "temperature" can adjust softmax sharpness:
```
Low temperature → sharper softmax → more concentrated attention → good for local information
High temperature → smoother softmax → more dispersed attention → good for long-range info (long-range doesn't need to be precise about which token anyway)
```

**YaRN's approach: NTK changes base + segmented scaling/adjustment for different dimension groups.** It is a common strong baseline, but not the "sole optimal" approach for all models and tasks. Later methods like LongRoPE and LongRoPE2 target even longer contexts.

- Fast dimensions: temperature = 1 (no adjustment, keep precision)
- Middle dimensions: smooth temperature transition
- Slow dimensions: slightly higher temperature (make long-range attention smoother)

In [ ]:
# YaRN's segmented strategy: divide dimensions into three groups by wavelength
import torch
import matplotlib.pyplot as plt
import math

scale, target_len = 4, 16384
pair_indices = torch.arange(0, 64, 2).float()
base_new = 10000 * (scale ** (64 / 62))
freqs_new = 1.0 / (base_new ** (pair_indices / 64))
wavelengths = 2 * math.pi / freqs_new  # positions needed for one full rotation

# Segmentation thresholds
low_bound = target_len / 1.0    # wavelength > this → low frequency (needs scaling)
high_bound = target_len / 4.0   # wavelength < this → high frequency (no scaling)

# ramping: smooth transition from 0 (no adjustment) to 1 (scale ×)
smooth = torch.clamp((wavelengths - high_bound) / (low_bound - high_bound), 0.0, 1.0)
dim_scale = (1 - smooth) * 1.0 + smooth * scale

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(32), dim_scale.numpy())
axes[0].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='No scaling')
axes[0].axhline(y=scale, color='red', linestyle='--', alpha=0.5, label=f'Scale {scale}x')
axes[0].set_xlabel('Dimension pair index (0=fast, 31=slow)'); axes[0].set_ylabel('Scale factor')
axes[0].set_title(f'YaRN: keep early pairs, scale later pairs {scale}x\nSmooth transition in the middle')
axes[0].legend()

axes[1].plot(wavelengths.numpy(), 'o-', markersize=3)
axes[1].axhline(y=high_bound, color='green', linestyle='--', alpha=0.5, label=f'High-frequency threshold ({high_bound:.0f})')
axes[1].axhline(y=low_bound, color='red', linestyle='--', alpha=0.5, label=f'Low-frequency threshold ({low_bound:.0f})')
axes[1].set_xlabel('Dimension pair index'); axes[1].set_ylabel('Wavelength (tokens per cycle)')
axes[1].set_yscale('log'); axes[1].set_title('Wavelength by dimension\nShort=high freq (unchanged), long=low freq (scaled)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# YaRN = NTK changes base + segmented smooth transition, a common strong baseline

## 11. Three Methods in One Sentence Each

| Method | One sentence | How it's done | Needs fine-tuning? |
|------|----------|----------|----------|
| **PI** | Divide all house numbers by 2 | Position index × scaling factor | Yes |
| **NTK** | Slow down the clock's rotation speed | Increase RoPE's base value | No |
| **YaRN** | NTK + segmented processing for different dimensions | Change base + segmented smooth transition | Usually no or minimal fine-tuning |

**Most important practical knowledge**: in most cases, you just need to increase `rope_theta` (the base value) in the model config file:
- 4K → 8K: change to around 500000
- 4K → 32K: change to around 1000000

That's exactly what LLaMA 3 did — changed base from 10000 to 500000, going directly from 8K to 32K.

In [ ]:
# Complete code: a RoPE module supporting three extrapolation strategies
import torch
import torch.nn as nn

class ExtrapolatableRoPE(nn.Module):
    """RoPE supporting PI / NTK / YaRN extrapolation strategies"""

    def __init__(self, d_k, max_seq_len=4096, base=10000, strategy='ntk'):
        super().__init__()
        self.d_k = d_k
        self.max_seq_len = max_seq_len
        self.base = base
        self.strategy = strategy
        self._update_cache(max_seq_len, base)

    def _update_cache(self, seq_len, base, pi_scale=1.0):
        """Recompute cos/sin cache"""
        positions = torch.arange(seq_len).float() / pi_scale
        freq = 1.0 / (base ** (torch.arange(0, self.d_k, 2).float() / self.d_k))
        angles = positions.unsqueeze(1) * freq.unsqueeze(0)
        cos = angles.cos().repeat_interleave(2, dim=-1)
        sin = angles.sin().repeat_interleave(2, dim=-1)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)

    def set_extrapolation(self, target_len):
        """Set extrapolation target length"""
        if target_len <= self.max_seq_len:
            return

        scale = target_len / self.max_seq_len

        if self.strategy == 'pi':
            self._update_cache(target_len, self.base, pi_scale=scale)
        elif self.strategy == 'ntk':
            new_base = self.base * (scale ** (self.d_k / (self.d_k - 2)))
            self._update_cache(target_len, new_base)
        elif self.strategy == 'yarn':
            new_base = self.base * (scale ** (self.d_k / (self.d_k - 2)))
            self._update_cache(target_len, new_base)  # Simplified; real YaRN also adjusts temperature

        print(f"Extrapolation: {self.max_seq_len} → {target_len} (strategy={self.strategy})")

    def forward(self, q, k, offset=0):
        """Apply rotation to Q and K"""
        seq_len = q.shape[-2]
        cos = self.cos[offset:offset + seq_len].to(q.device)
        sin = self.sin[offset:offset + seq_len].to(q.device)

        q_rot = q * cos + (torch.stack([-q[..., 1::2], q[..., ::2]], dim=-1).flatten(-2) * sin)
        k_rot = k * cos + (torch.stack([-k[..., 1::2], k[..., ::2]], dim=-1).flatten(-2) * sin)
        return q_rot, k_rot

# Test
rope = ExtrapolatableRoPE(d_k=64, max_seq_len=4096, strategy='ntk')
rope.set_extrapolation(32768)

q = torch.randn(1, 1, 100, 64)
k = torch.randn(1, 1, 100, 64)
q_rot, k_rot = rope(q, k)
print(f"Q: {q.shape} → after rotation: {q_rot.shape}")

## 12. Methods for Validating Long Context

You've extended the context from 4K to 32K, but how do you prove it actually "understands" long text? You can't just go by feel.

**Probe test = design a question that can only be answered correctly by understanding the full text.** If the model answers correctly on a long text, it means its attention to distant tokens is effective.

#### 12.1 Needle in a Haystack — The Classic Test

The procedure is straightforward:
1. Generate a large body of irrelevant text (the "haystack"), say a 32K-token article
2. Insert a sentence at some position (the "needle"), like "the password is 12345"
3. Ask the model: "What is the password?"
4. If the model can find and correctly answer the password from the 32K text → attention at that position is working well

Place the needle at different positions (beginning, middle, end), use different text lengths (1K, 2K, 4K, ..., 32K), test every combination → draw a heatmap.

In [ ]:
# Needle in a Haystack test matrix (simulated): different lengths × different positions
context_lengths = [1024, 2048, 4096, 8192, 16384, 32768]
positions = [0.0, 0.25, 0.5, 0.75, 1.0]  # needle position (0=beginning, 1=end)

# Simulated results: ✅=correct, ❌=incorrect
results = [
    [True,  True,  True,  True,  True ],   # 1K
    [True,  True,  True,  True,  True ],   # 2K
    [True,  True,  True,  True,  True ],   # 4K
    [True,  True,  True,  True,  True ],   # 8K
    [True,  True,  False, True,  True ],   # 16K — lost the middle
    [True,  False, False, True,  True ],   # 32K — lost front-middle too
]

print("Needle in a Haystack test matrix:")
print(f"{'Length':<8}", *[f"{p:.0%} pos" for p in positions])
for i, cl in enumerate(context_lengths):
    print(f"{cl:<8}", *[f"{'✅' if r else '❌'}   " for r in results[i]])
# Lost in the Middle: needles in the middle are more likely to be lost

#### 12.2 Tougher than Needle in a Haystack: RULER

Needle in a Haystack only tests "find one sentence" ability. RULER is more comprehensive:

| Test | What it does | Why it's harder |
|------|--------|----------|
| **Multi-needle recall** | Hide 3 different pieces of information, ask 3 times | Requires maintaining attention at multiple locations simultaneously |
| **Multi-hop reasoning** | Beginning says A=1, end says B=A+1, ask B | Requires combining two distant pieces of information to reason |
| **Variable tracking** | Track a value through multiple changes in the text | Requires updating memory |
| **Word frequency counting** | Count how many times a word appears in the full text | Requires traversing the full text and counting |

**Multi-hop reasoning is the most revealing:**
```
Text at 10% position: "Company A's revenue is 10 billion"
Text at 90% position: "Company B's revenue is 2× A's"
Question: "What is Company B's revenue?"

Model needs to:
  1. Find the information at 10% → 10 billion
  2. Find the information at 90% → 2×
  3. Combine and reason → 20 billion
```
This is much harder than simply "finding one sentence," because it requires the model to maintain precise attention at both the beginning and the end simultaneously.

In [ ]:
# Multi-hop reasoning probe example: needs to remember multiple distant pieces of information simultaneously
print("=== Multi-hop reasoning probe (simulated) ===")
print(f"Text: 32K tokens")
print(f"  5%  position: Apples are 5 yuan each")
print(f"  50% position: Xiaoming buys 3")
print(f"  95% position: Spend 10+ get 2 off")
print(f"  Question: How much does Xiaoming pay? → Answer: 5×3=15, spend 10+ get 2 off = 13 yuan")
print(f"\nRequires 5%→50%→95% three hops; missing any one gives wrong answer")
# Answer 15 yuan → missed the discount info at the end
# Answer "don't know" → didn't find any information

#### 12.3 PPL Curves — Directly See "How Confused the Model Is"

**Perplexity (PPL)** = how confused the model is. Lower is better; lower means the model is more "confident" about the next token.

The best test: feed the same long text to the model and see whether PPL suddenly spikes at the training window boundary (4096).

In [ ]:
# Simulated PPL curves: how different extrapolation methods behave beyond the window boundary
import torch
import matplotlib.pyplot as plt

lengths = torch.linspace(512, 32768, 100)
ppl_none, ppl_pi, ppl_ntk = [], [], []

for L in lengths:
    if L <= 4096:
        ppl_none.append(10.0); ppl_pi.append(10.0); ppl_ntk.append(10.0)
    else:
        over = (L - 4096) / 4096
        ppl_none.append(10 + 100 * over**1.5)   # Explodes directly
        ppl_pi.append(10 + 8 * over**0.8)         # Rises slowly
        ppl_ntk.append(10 + 3 * over**0.5)        # Barely rises

plt.figure(figsize=(10, 5))
plt.plot(lengths.numpy(), ppl_none, label='No extrapolation', linewidth=2, color='red')
plt.plot(lengths.numpy(), ppl_pi, label='PI', linewidth=2, color='orange')
plt.plot(lengths.numpy(), ppl_ntk, label='NTK / YaRN', linewidth=2, color='green')
plt.axvline(x=4096, color='gray', linestyle='--', linewidth=1.5,
            alpha=0.7, label='Training window boundary (4K)')
plt.xlabel('Context length (tokens)'); plt.ylabel('PPL (lower is better)')
plt.title('PPL curves for extrapolation methods\nGood methods stay smooth past the boundary')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
# ✅ Good extrapolation: PPL stays smooth past 4096  |  ❌ Bad: PPL explodes violently

#### 12.4 Lost in the Middle — A Problem All Models Have

Even with perfect extrapolation, there's one problem that remains unsolved: **models naturally pay more attention to the beginning and end of text, while neglecting the middle.**

This is called the **"Lost in the Middle"** phenomenon.

Why? Because attention's softmax makes weights "compete" to sum to 1. The beginning and end have structural advantages:
- Beginning tokens are seen by all subsequent tokens (the starting point of causal attention)
- End tokens are closest to the current generation position (recency bias)
- Middle tokens are disadvantaged on both sides

In [ ]:
# Visualize "Lost in the Middle": middle information is naturally ignored
import torch
import matplotlib.pyplot as plt

seq_len = 8192; target_pos = 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ✅ Ideal: attention focuses on the key position
ideal_attn = torch.zeros(seq_len)
ideal_attn[target_pos] = 0.8
ideal_attn[max(0, target_pos-20):target_pos+20] += 0.01
ideal_attn /= ideal_attn.sum()
axes[0].plot(ideal_attn.numpy(), linewidth=0.5, color='green')
axes[0].axvline(x=target_pos, color='red', linestyle='--', alpha=0.7, label=f'Key position ({target_pos})')
axes[0].set_title('Ideal: attention focuses on target'); axes[0].legend(); axes[0].grid(True, alpha=0.2)

# ⚠ Reality: Lost in the Middle → attention at edges, low in middle
u_shape = 1.0/(1+torch.arange(seq_len).float()) + 1.0/(1+torch.arange(seq_len-1,-1,-1).float())
u_shape /= u_shape.sum()
axes[1].plot(u_shape.numpy(), linewidth=0.5, color='purple')
axes[1].axvline(x=target_pos, color='red', linestyle='--', alpha=0.7, label=f'Key position ({target_pos})')
axes[1].set_title('Reality: Lost in the Middle'); axes[1].legend(); axes[1].grid(True, alpha=0.2)

# Recall rate: U-shaped curve
positions = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
recall_rates = [0.95, 0.85, 0.60, 0.40, 0.55, 0.85, 0.98]
axes[2].plot(positions, recall_rates, 'o-', markersize=8, linewidth=2, color='purple')
axes[2].set_xlabel('Information position'); axes[2].set_ylabel('Recall')
axes[2].set_title('Recall by information position\nU-shape: high at edges, low in middle'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
# This is a structural problem, not something changing RoPE can fix. In practice, put important information at the edges.

## 13. Engineering Reality: Long Context is Not Just an Algorithm Problem

Even with perfect extrapolation algorithms, there's a hard bottleneck: **GPU memory**.

Recall the KV Cache from Side Quest 14: for each new token generated, the K and V of all previous tokens must be stored. The longer the sequence, the larger the KV Cache:

```
4K context  → KV Cache ≈ 2GB   (one consumer GPU is enough)
32K context → KV Cache ≈ 16GB  (A100 barely)
128K context → KV Cache ≈ 64GB  (needs multi-GPU)
1M context  → KV Cache ≈ 500GB (requires special techniques)
```

Engineering solutions:
- **KV Cache quantization**: store K and V in 8-bit or even 4-bit, saving 2–4× memory
- **Ring Attention**: split a long sequence into segments, distribute across multiple GPUs, each GPU handles one segment
- **StreamingLLM**: keep a few "anchor" tokens at the beginning + the most recent batch of tokens, discard the middle

### 13.1 Sliding Window Attention — Limiting Attention Range to Reduce Computation

PI, NTK, and YaRN solve the position encoding problem: they let the model represent positions beyond the training window. But there's a more direct problem: even with perfect position encoding, the computation of standard causal attention itself makes long sequences impractical.

In standard causal attention, each token attends to all previous tokens. With sequence length N, the attention matrix is N×N, and both computation and KV Cache grow as N². When N=128K, a single layer's attention matrix has 16B elements, exceeding 64GB in FP32 — and that's just one layer. No matter how good the position encoding is, computation and storage at this scale are unacceptable.

Sliding Window Attention takes a direct approach: each token no longer attends to all history, but only looks at the most recent W tokens. W is the window size, a fixed constant (e.g., 4096). This changes computation from O(N²) to O(W·N) — growing linearly with sequence length instead of quadratically.

With W=4096 and N=128K, standard attention requires 16B dot products while Sliding Window needs only 524M — a 97% reduction. KV Cache also shrinks accordingly: each layer only stores K and V for the most recent 4096 tokens instead of all 128K.

Implementation-wise, it adds one extra step when constructing the attention mask: in addition to the causal mask (not seeing the future), add a distance mask — if the key's position is more than W-1 away from the query, set it to -inf. softmax naturally outputs 0 for -inf, equivalent to not attending.

The cost is losing long-range attention — token 100000 cannot see token 0, even if token 0 contains critical information. Mistral 7B is a classic example: the official documentation and model description emphasize sliding window attention with a window size of 4096. Other long-context models also mix global attention, chunk attention, RingAttention, or retrieval-based memory; the specific window size and layer allocation must be checked in each model's config, not guessed from general experience.

Sliding Window and RoPE extrapolation solve two independent problems: RoPE makes the model "recognize" distant positions, while Sliding Window makes the model "able to compute" sequences that long. They are typically used together.

In [ ]:
import torch
import matplotlib.pyplot as plt

def create_sliding_window_mask(seq_len, window_size):
    """
    Construct a Sliding Window Attention mask.

    Each token i can only attend to tokens in range [i - window_size + 1, i],
    while preserving the causal mask (cannot see the future).

    Returns: [seq_len, seq_len], 0 = allowed, -inf = blocked
    """
    # Standard causal mask (upper triangle = -inf)
    causal_mask = torch.triu(
        torch.full((seq_len, seq_len), float('-inf')), diagonal=1
    )

    # Find historical positions beyond the window
    row = torch.arange(seq_len).unsqueeze(1)  # [seq_len, 1]
    col = torch.arange(seq_len).unsqueeze(0)  # [1, seq_len]
    distance = row - col                      # [seq_len, seq_len]
    outside_window = distance > window_size - 1

    # Merge causal + sliding window
    mask = causal_mask.clone()
    mask[outside_window] = float('-inf')

    return mask

# Demo: masks with different window sizes
seq_len = 12
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax_idx, (w, title) in enumerate([
    (12, "Standard Causal Attention\n(W = N = 12)"),
    (6,  "Sliding Window\n(W = 6)"),
    (3,  "Sliding Window\n(W = 3)"),
]):
    mask = create_sliding_window_mask(seq_len, w)
    ax = axes[ax_idx]
    # Green = visible, red = invisible
    im = ax.imshow(mask, cmap='RdYlGn_r', aspect='auto', vmin=-10, vmax=0)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")

plt.tight_layout()
plt.show()

# Computation comparison
print("=== Sliding Window Computation Comparison ===")
print()
print(f"{'N':>8s}  {'Standard O(N²)':>14s}  {'SW O(W·N)':>14s}  {'Reduction':>8s}")
print("-" * 50)
for N in [4096, 8192, 32768, 131072]:
    W = 4096
    full_ops = N * N
    sw_ops = W * N
    reduction = (1 - sw_ops / full_ops) * 100
    print(f"{N:>8d}  {full_ops:>14,d}  {sw_ops:>14,d}  {reduction:>7.1f}%")

print()
print("Key observations:")
print("1. With fixed window size W, computation grows linearly with N (O(W·N)), not quadratically")
print("2. When N >> W, the computation savings are very significant")
print("3. Sliding Window also saves KV Cache: each layer stores only W K,V pairs instead of N")
print("4. The cost is losing long-range attention — occasional global attention layers are needed to compensate")

## 14. Hands-on: Extending 4K to 32K

```
Step 1: Choose a method
  → Don't want to train → NTK-aware (just change rope_theta)
  → Willing to fine-tune → YaRN (slightly better results)

Step 2: Change parameters
  → Find rope_theta in the model's config.json
  → 4K→32K reference value: change to 500000 ~ 1000000
  → Or calculate by formula: new base = 10000 × (8)^(64/62) ≈ 86000

Step 3: Test
  → Needle in a Haystack full-position heatmap
  → RULER multi-hop reasoning
  → PPL curve (should be smooth at the boundary)

Step 4: If not good enough
  → Middle position recall is poor → adjust YaRN's segmentation parameters
  → Overall too high → fine-tune on a small amount of long-text data
  → Not enough GPU memory → use KV Cache quantization + vLLM
```

## 15. Hands-on: ModelScope + NTK Extension

We've covered a lot of theory; now let's do it for real.

**Task**: Pull a Qwen model from ModelScope, check its default context configuration, calculate the `rope_theta` needed for extension using the NTK-aware method, then run a Needle in a Haystack test to verify whether the extension works.

In [ ]:
# === Optional hands-on dependency: transformers / modelscope ===
# The first half of this notebook is pure hand-written principle demos; this section
# automatically falls back to ToyModel if dependencies are missing.

import torch

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from modelscope import snapshot_download
    HAS_REAL_LONG_CONTEXT_DEMO = True
except ModuleNotFoundError as e:
    AutoModelForCausalLM = AutoTokenizer = snapshot_download = None
    HAS_REAL_LONG_CONTEXT_DEMO = False
    print(f"Optional dependency missing: {e}")
    print("Using ToyTokenizer/ToyModel to run through the rest; for real Qwen hands-on, install transformers + modelscope.")

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# === Download and load Qwen2.5-0.5B-Instruct from ModelScope ===
# If optional dependencies or network are unavailable locally, use ToyModel to keep the notebook runnable.

import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

if HAS_REAL_LONG_CONTEXT_DEMO:
    print("Downloading model from ModelScope...")
    model_dir = snapshot_download(model_name, revision="master")
    print(f"Model downloaded to: {model_dir}\n")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True,
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    model.eval()
else:
    print("Skipping real model download, creating offline ToyModel for data flow demo.")

    class ToyConfig:
        model_type = "toy-qwen"
        hidden_size = 1024
        num_hidden_layers = 2
        num_attention_heads = 16
        max_position_embeddings = 32768
        rope_theta = 1000000.0
        rope_scaling = None

    class ToyTokenizer:
        def __init__(self):
            self.eos_token_id = 0
            self.vocab = {"<eos>": 0}
            self.reverse = {0: ""}

        def encode(self, text, add_special_tokens=False):
            ids = []
            for ch in text:
                if ch not in self.vocab:
                    self.vocab[ch] = len(self.vocab)
                    self.reverse[self.vocab[ch]] = ch
                ids.append(self.vocab[ch])
            return ids

        def decode(self, ids, skip_special_tokens=False):
            return "".join(self.reverse.get(int(i), "") for i in ids)

        def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
            text = "".join(f"[{m['role']}] {m['content']}\n" for m in messages)
            if add_generation_prompt:
                text += "[assistant] "
            return self.encode(text, add_special_tokens=False) if tokenize else text

    class ToyModel:
        def __init__(self, tokenizer):
            self.config = ToyConfig()
            self.tokenizer = tokenizer

        def to(self, device):
            return self

        def eval(self):
            return self

        def generate(self, input_tensor, max_new_tokens=50, **kwargs):
            suffix = torch.tensor([self.tokenizer.encode("8842")], device=input_tensor.device)
            return torch.cat([input_tensor, suffix], dim=1)

    tokenizer = ToyTokenizer()
    model = ToyModel(tokenizer).to(DEVICE).eval()

print(f"Model type: {model.config.model_type}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Attention heads: {model.config.num_attention_heads}")
print(f"max_position_embeddings: {model.config.max_position_embeddings}  ← max position during training")
print(f"rope_theta: {model.config.rope_theta}  ← RoPE base value")
print(f"rope_scaling: {model.config.rope_scaling}  ← whether extrapolation strategy is enabled")

#### 15.1 Calculating rope_theta for NTK Extension

NTK formula: `new base = old base × scale^(d/(d-2))`

Where `scale = target length / original length`.

Below we calculate rope_theta for several common extension targets using Qwen2.5-0.5B:

In [ ]:
# === Use NTK formula to calculate rope_theta for different target lengths ===
# Qwen2.5-0.5B default: max_position = 32768, rope_theta = 1000000

def ntk_rope_theta(original_base, original_len, target_len, d_k):
    """NTK-aware: calculate rope_theta after extension"""
    scale = target_len / original_len
    new_base = original_base * (scale ** (d_k / (d_k - 2)))
    return new_base

# Qwen2.5-0.5B parameters
d_k = model.config.hidden_size // model.config.num_attention_heads  # head_dim
original_base = model.config.rope_theta
original_len = model.config.max_position_embeddings

print(f"Current config:")
print(f"  head_dim = {d_k}")
print(f"  rope_theta = {original_base:,}")
print(f"  max_position = {original_len:,} tokens")
print()

# Calculate rope_theta for extending to different lengths
targets = {
    "64K": 65536,
    "128K": 131072,
    "256K": 262144,
    "1M": 1048576,
}

print("NTK extension table:")
print(f"{'Target':<10} {'scale':<10} {'New rope_theta':<15} {'Formula'}")
print("-" * 65)
for label, target in targets.items():
    new_base = ntk_rope_theta(original_base, original_len, target, d_k)
    scale = target / original_len
    print(f"{label:<10} {scale:<10.1f} {new_base:<15,.0f} base×{scale:.1f}^({d_k}/{d_k-2})")

print(f"\nOperation: modify rope_theta in config.json to the corresponding value")
print(f"  Example: 4K→128K: change rope_theta from {original_base:,} to {ntk_rope_theta(original_base, original_len, 131072, d_k):,.0f}")

#### 15.2 Needle in a Haystack Test: Verifying Long Context Actually Works

Now construct a long text, hide a sentence in it, and see if the model can find it.

**Test design**:
1. Generate a "haystack" — fill with irrelevant text up to target length
2. Insert a "needle" at a specified position — a key piece of information
3. Ask the model a question that can only be answered by reading the needle
4. See if the model can answer correctly

In [ ]:
# === Needle in a Haystack test ===

import torch

def build_needle_haystack(tokenizer, target_len, needle, needle_pos, question):
    """
    Construct a Needle in a Haystack test

    Args:
        target_len: target total token count
        needle: the information to hide (string)
        needle_pos: needle position (0~1, 0=beginning, 1=end)
        question: the question to ask
    """
    # Haystack: fill with a looping irrelevant text
    haystack_sentence = (
        "The quick brown fox jumps over the lazy dog. "
        "Machine learning is a subset of artificial intelligence. "
        "The weather today is quite pleasant with a gentle breeze blowing. "
        "Many people enjoy reading books and drinking coffee in the morning. "
    )

    # Encode haystack text to see how many tokens per sentence
    haystack_tokens = tokenizer.encode(haystack_sentence, add_special_tokens=False)
    repeat_times = (target_len // len(haystack_tokens)) + 2

    # Construct full text: haystack + needle insertion + haystack
    repeat_text = haystack_sentence * repeat_times
    full_tokens = tokenizer.encode(repeat_text, add_special_tokens=False)

    # Calculate insertion position (token level)
    insert_idx = int(target_len * needle_pos)
    needle_tokens = tokenizer.encode(f"\n\n[Important info]: {needle}\n\n", add_special_tokens=False)

    # Concatenate
    prefix = full_tokens[:insert_idx]
    suffix = full_tokens[insert_idx:target_len - len(needle_tokens)]
    test_tokens = prefix + needle_tokens + suffix
    test_tokens = test_tokens[:target_len]

    # Construct chat prompt
    test_text = tokenizer.decode(test_tokens)
    messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts information. Answer briefly based on the text above."},
        {"role": "user", "content": f"Please read the following text and answer the question.\n\n{test_text}\n\nQuestion: {question}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt, tokenizer.encode(prompt, add_special_tokens=False)

# Test parameters
target_len = 8000  # 8K token test text (within the default 32K window)
needle = "The combination to open the safe is 8842"
needle_pos = 0.5  # Place in the middle
question = "What is the combination to the safe?"

prompt, input_ids = build_needle_haystack(tokenizer, target_len, needle, needle_pos, question)
print(f"Test text length: {len(input_ids)} tokens")
print(f"Needle position: {needle_pos*100:.0f}% (approximately token {int(target_len*needle_pos)})")
print(f"Needle content: '{needle}'")
print(f"Question: '{question}'")
print()

# Generate answer
input_tensor = torch.tensor([input_ids]).to(DEVICE)
with torch.no_grad():
    output = model.generate(
        input_tensor,
        max_new_tokens=50,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(output[0][len(input_ids):], skip_special_tokens=True)
print(f"Model answer: {answer.strip()}")
print()

# Check if correct
if "8842" in answer or "8842" in answer.replace(" ", ""):
    print("✅ Test passed! Model successfully found the needle in 8K context")
else:
    print("❌ Test failed! Model did not correctly find the needle information")

#### 15.3 Full Position Scan: Needle in a Haystack Heatmap

Needle in a Haystack is the standard method for testing long-context capability. The procedure is to prepare a very long text (haystack), hide a fact (needle) at different depth positions — for example, "the magic number is 78921" — then at the end ask "what is the magic number?" If the model answers correctly, it means it can attend to information at that depth.

A single test only gives one data point. A more thorough evaluation tests each position — e.g., at depths 0%, 25%, 50%, 75%, 100% — then plots the correctness at each position as a heatmap. The heatmap can intuitively show where the model is most likely to lose information: if the upper-left corner is blank and the lower-right is all green, the model isn't paying enough attention to information at the beginning; if the middle region is weak, there's a blind spot in attention. Below we draw this model's heatmap.

In [ ]:
# === Full-position Needle in a Haystack test ===

import torch

def test_needle_at_position(target_len, needle, needle_pos, question):
    """Run one needle test at a specified position, return whether successful"""
    prompt, input_ids = build_needle_haystack(tokenizer, target_len, needle, needle_pos, question)
    input_tensor = torch.tensor([input_ids]).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            input_tensor,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(output[0][len(input_ids):], skip_special_tokens=True)
    # Loose match: just need to produce the key number
    return "8842" in answer.replace(" ", "")


# Test at different positions within the default 32K window
positions = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
results = []

print(f"Needle in a Haystack test: text length={target_len} tokens")
print(f"{'Position':<12} {'Result':<8} {'Notes'}")
print("-" * 40)

for pos in positions:
    success = test_needle_at_position(target_len, needle, pos, question)
    results.append(success)
    desc = ""
    if pos < 0.2:
        desc = "(beginning — easily visible)"
    elif pos > 0.8:
        desc = "(end — recency bias)"
    else:
        desc = "(middle — Lost in Middle high-risk zone)"
    print(f"{pos*100:3.0f}% pos    {'✅' if success else '❌'}      {desc}")

print()
success_rate = sum(results) / len(results)
print(f"Success rate: {success_rate:.0%} ({sum(results)}/{len(results)})")

# If middle positions fail, it confirms Lost in the Middle
mid_results = [r for p, r in zip(positions, results) if 0.15 < p < 0.85]
edge_results = [r for p, r in zip(positions, results) if p <= 0.15 or p >= 0.85]
if sum(mid_results) < len(mid_results):
    print(f"\n⚠ Middle position failures → confirms Lost in the Middle phenomenon")
    print(f"   Edge success rate: {sum(edge_results)}/{len(edge_results)}")
    print(f"   Middle success rate: {sum(mid_results)}/{len(mid_results)}")

#### 15.4 Hands-on Summary

Through the demos above, we completed a full long-context extension verification workflow:

1. **Get model from ModelScope** → `snapshot_download` + `AutoModelForCausalLM`
2. **Check default configuration** → `rope_theta` and `max_position_embeddings` are the key parameters
3. **Calculate NTK extension** → one formula `new_base = old_base × scale^(d/(d-2))`
4. **Modify configuration** → change `rope_theta` in `config.json`; most cases don't require retraining
5. **Needle in a Haystack verification** → full-position testing at target length, confirming the model can correctly recall information

**Production deployment steps**:
```bash
# 1. Modify config.json in the model directory
# Find "rope_theta": 1000000.0
# Change to the calculated new value, e.g., "rope_theta": 10000000.0

# 2. Use with inference framework
# vLLM: add --max-model-len 131072 at launch
# Transformers: directly load the modified config

# 3. Verify
# Run full-position Needle in a Haystack + PPL curve
```

**Key insight**: NTK's `rope_theta` change is just mathematical frequency compression — it doesn't change any model weights. It works because RoPE's frequency-domain structure naturally supports this kind of compression. This is RoPE's greatest advantage over learned position encodings (GPT-2) and sinusoidal encodings (original Transformer).

## Summary

1. ✅ **Extrapolation** = using patterns learned within the training window to handle positions beyond the window
2. ✅ **RoPE** = encodes positions via rotation; different dimensions rotate at different speeds (fast like second hand, slow like hour hand)
3. ✅ Reason for extrapolation failure = slow dimensions don't complete a full rotation within the training window; at inference, angle values exceed the training range
4. ✅ **PI** = proportionally compress all position indices into the training range (one-size-fits-all, loses precision)
5. ✅ **NTK** = only change the base value, automatically compressing slow dimensions more and fast dimensions less (elegant)
6. ✅ **YaRN** = NTK + segmented smooth processing; a common strong baseline
7. ✅ Extrapolation methods in practice only require changing one parameter, `rope_theta`; most cases don't need retraining
8. ✅ Probe tests: Needle in a Haystack (single needle), RULER (multi-needle multi-hop), PPL curves (check boundary smoothness)
9. ✅ **Lost in the Middle** = middle positions are naturally prone to being ignored → put important information at the edges
10. ✅ Engineering also has KV Cache memory bottleneck, requiring quantization/RingAttention and other auxiliary methods

**One-sentence summary**: Long context extrapolation = exploit the fact that RoPE dimensions rotate at different speeds; slow down the slow ones (prevent them from exceeding the training range), keep the fast ones (maintain local precision). NTK-aware achieves this by only changing `rope_theta`; YaRN adds segmented smooth transitions on top; for even longer contexts, see LongRoPE / LongRoPE2. Use Needle in a Haystack and PPL curves to verify effectiveness. References: [YaRN](https://arxiv.org/abs/2309.00071), [LongRoPE](https://arxiv.org/abs/2402.13753), [LongRoPE2](https://arxiv.org/abs/2502.20082), [Mistral 7B documentation](https://docs.mistral.ai/models/mistral-7b-0-2).

## Exercises

To be added